In [ ]:
import numpy as np
from scipy.fft import fft, ifft, fft2, ifft2, fftfreq, fftshift

In [ ]:
kappa = 0.0025**2

Lx, Ly = 0.5, 0.5                                   # Definerer området
Nx, Ny = 256, 256                                   # Definerer oppløsning
dA = Lx/Nx * Ly/Ny                                  # Definerer arealelement
x, y = np.linspace(0, Lx, Nx, endpoint = False), np.linspace(0, Ly, Ny, endpoint = False)     
X, Y = np.meshgrid(x, y)                            # Oppretter grid

# Definerer liste med to ulike tidsintervall
t0 = 0
intervals = np.array([4.0, 0.01])                        

# Definerer liste med to ulike initialbetingelser
rng = np.random.default_rng(12345)
noise_1 = 0.05
u0_base_1 = 0.0
U0_1 = np.ones_like((Ny, Nx))
U0_1 = np.full((Ny, Nx), u0_base_1) +  noise_1*rng.standard_normal((Ny, Nx))

noise_2 = 0.05
u0_base_2 = -0.45
U0_2 = np.ones_like((Ny, Nx))
U0_2 = np.full((Ny, Nx), u0_base_2) +  noise_2*rng.standard_normal((Ny, Nx))

U0s = np.array([U0_1, U0_2])

# Definerer liste med to ulike steglengder
dts = np.array[10**(-3), 10**(-4)]                  

In [ ]:
# Definerer nødvendige funksjoner

def total_mass(U):
    return np.sum(U)*dA


def F(u):                               # Definerer funksjon for Helmholtz frie energitetthet F(u)
    return (1/4)*(u**2 - 1)**2

def mixing_energy(U):                   # Definerer funksjon for blandingsenergi
    return np.sum(F(U))*dA              # Evaluerer F(u) i U og summerer over hele gridet


def interface_energy(kappa, U, Nx, Ny, Lx, Ly):
    U_hat = fft2(U)

    kx = fftfreq(Nx, d=Lx/Nx)*2*np.pi
    ky = fftfreq(Ny, d=Ly/Ny)*2*np.pi
    KX, KY = np.meshgrid(kx, ky, sparse=True)

    K2 = KX**2 + KY**2

    U_grad_squared = ifft2(K2 * np.abs(U_hat)**2).real
    
    int_energy = np.sum((kappa/2)*U_grad_squared) * dA

    return int_energy


In [ ]:
for T, U0, dt in zip(intervals, U0s, dts):

    solver = cahn_hilliard_backward_euler(kappa = kappa, 
                                    X = X, Y = Y, U0 = U0, 
                                    t0 = t0, T = T, Nt = T/dt,
                                    g = g, 
                                    alpha=1.5)

    for U_hat, t in solver:
        U = ifft2(U_hat).real
        